# Benchmark
Comparison of performance and accuracy between the methods in the `numerical_methods` library.

## Cell 1 – Imports

In [12]:
import numpy as np
import sympy as sp
from time import perf_counter
from scipy.integrate import quad, dblquad
from scipy.optimize._numdiff import approx_derivative
from scipy.optimize import brentq, approx_fprime

# Imported library already installed with pip.
import numerical_methods as nm

## Cell 2 – Helpers
Every section below times a method, compares it to a reference value, and prints one row.
These three functions do that job once so it isn't rewritten in each section.

In [13]:
def timed(func, *args, **kwargs):
    """Run func and return (result, elapsed_time_ms)."""
    start = perf_counter()
    result = func(*args, **kwargs)
    elapsed = (perf_counter() - start) * 1000
    return result, elapsed


def print_header(value_label = "Result", method = "Method"):
    print(f"{method:30}{value_label:>15}{'Error':>15}{'Time (ms)':>15}")
    print("-" * 75)


def print_row(name, value, error, elapsed):
    """Print one formatted benchmark row. Pass value=None to show '—'."""
    value_str = f"{value:15.6f}" if value is not None else f"{'—':>15}"
    print(f"{name:30}{value_str}{error:15.2e}{elapsed:15.4f}")

## Cell 3 – Available methods
Only integration and differentiation methods are listed as dictionaries here: every function in each of these two families shares the same call signature (`method(f, a, b, n)` and `method(f, x, h)`), so a single loop can call any of them.

Root finding, series approximation, and linear algebra methods each take a different combination of arguments, so a shared dictionary would need one-off wrapping anyway — those calls are defined locally in their own cells, right next to the parameters they use.

In [14]:
integration_methods = {
    "Rectangle Rule": nm.rectangle_integrate,
    "Midpoint Method": nm.midpoint_integrate,
    "Trapezoid Rule": nm.trapezoidal_integrate,
    "First Simpson Rule": nm.simpson1_integrate,
    "Second Simpson Rule": nm.simpson2_integrate,
    "Gauss-Legendre Quadrature": nm.gauss_legendre_integrate,
    "Monte Carlo": nm.monte_carlo_integrate,
}

differentiation_methods = {
    "Forward Difference": nm.fd_forward_derivative,
    "Backward Difference": nm.fd_backward_derivative,
    "Central Difference": nm.fd_central_derivative,
    "Central Difference nth": nm.fd_nth_derivative,
    "Richardson Method": nm.richardson_derivative,
}

## Function of a variable 'f' that will be used throughout the benchmark.

In [15]:
f = lambda x: np.cos(x) ** 2 + np.sin(2 * x)

## Cell 4 – Numerical Integration

In [16]:
# Simple Integral
a, b = -np.pi, np.pi
n = 120  # interval subdivisions (sample count for Monte Carlo)
exact = np.pi

print_header()

for name, method in integration_methods.items():
    result, elapsed = timed(method, f, a, b, n)
    print_row(name, result, nm.error_calculate(exact, result), elapsed)

reference, elapsed_ref = timed(lambda: quad(f, a, b)[0])
print_row("SciPy (quad)", reference, nm.error_calculate(exact, reference), elapsed_ref)

print("")

# Double Integral
nx = ny = 1024
F = lambda x, y: x ** 2 * y
exact = 2.0 / 3.0

print_header()

result1, elapsed1 = timed(nm.midpoint_double_integrate, F, 0, 1, 0, 2, nx, ny)
result2, elapsed2 = timed(nm.trapezoidal_double_integrate, F, 0, 1, 0, 2, nx, ny)

reference, elapsed_ref = timed(lambda: dblquad(lambda y, x: F(x, y), 0, 1, lambda x: 0, lambda x: 2)[0])

print_row("Midpoint Double", result1, nm.error_calculate(exact, result1), elapsed1)
print_row("Trapezoidal Double", result2, nm.error_calculate(exact, result2), elapsed2)
print_row("SciPy (dblquad)", reference, nm.error_calculate(exact, reference), elapsed_ref)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Rectangle Rule                       3.141593       2.83e-16         0.1066
Midpoint Method                      3.141593       0.00e+00         0.0783
Trapezoid Rule                       3.141593       0.00e+00         0.2146
First Simpson Rule                   3.141593       0.00e+00         0.1165
Second Simpson Rule                  3.141593       0.00e+00         0.1035
Gauss-Legendre Quadrature            3.141593       6.50e-15         3.2202
Monte Carlo                          3.337354       6.23e-02         0.1689
SciPy (quad)                         3.141593       1.41e-16         0.0341

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Midpoint Double                      0.666667       2.38e-07       261.8831
Trapezoidal

## Cell 5 – Numerical Differentiation

In [17]:
evaluation_point = 2
delta_x = 0.0001

x_sym = sp.symbols('x')
f_sym = sp.cos(x_sym) ** 2 + sp.sin(2 * x_sym)
df_sym = sp.diff(f_sym, x_sym)
reference = float(df_sym.subs(x_sym, evaluation_point))

print_header()

for name, method in differentiation_methods.items():
    result, elapsed = timed(method, f, evaluation_point, delta_x)
    print_row(name, result, nm.error_calculate(reference, result), elapsed)

# SciPy: approx_fprime(x, f, h) -> gradient
result, elapsed = timed(approx_fprime, np.array([evaluation_point]), lambda v: f(v[0]), delta_x)
print_row("SciPy (approx_fprime)", result[0], nm.error_calculate(result[0], reference), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Forward Difference                  -0.550268       3.94e-04         0.0425
Backward Difference                 -0.550701       3.94e-04         0.0054
Central Difference                  -0.550485       6.67e-09         0.0028
Central Difference nth              -0.550485       1.67e-09         0.0076
Richardson Method                   -0.550485       5.03e-13         0.0052
SciPy (approx_fprime)               -0.550268       3.94e-04         0.1981


## Cell 6 – Root Finding

In [18]:
root_methods = [
    ("Bisection", lambda: nm.bisection_calculate(f, *root_interval, tol)),
    ("Newton-Raphson", lambda: nm.newton_raphson_calculate(f, x0_root, max_iter, tol)[0]),
    ("Ridders", lambda: nm.ridders_calculate(f, *root_interval, max_iter, tol)[0]),
    ("Brent (SciPy)", lambda: brentq(f, *root_interval, xtol=tol)),
]

# f has a sign change between -0.5 and -0.4 (bracket for Bisection / Ridders)
root_interval = (-0.5, -0.4)
x0_root = -0.45
max_iter = 100
tol = 1e-6

reference = float(sp.nsolve(f_sym, x_sym, x0_root))

print_header()

for name, method in root_methods:
    root, elapsed = timed(method)
    print_row(name, root, nm.error_calculate(reference, root), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Bisection                           -0.463648       5.03e-07         0.0417
Newton-Raphson                      -0.463648       8.45e-09         0.0151
Ridders                             -0.463648       3.91e-09         0.0140
Brent (SciPy)                       -0.463648       3.42e-09         0.0328


## Cell 7 – Series Approximation

In [19]:
series_methods = [
    ("Taylor Series", lambda: nm.taylor_approx(f, evaluation_x, expansion_point, order)),
    ("Fourier Series", lambda: nm.fourier_approx(f, evaluation_x, b, order)),
]

expansion_point = 0
order = 6
evaluation_x = 1.0
reference = f(evaluation_x)

print_header()
for name, method in series_methods:
    result, elapsed = timed(method)
    print_row(name, result, nm.error_calculate(reference, result), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Taylor Series                        1.222223       1.75e-02         0.0997
Fourier Series                       1.201224       1.85e-16         1.4738


## Cell 8 – Linear Algebra

In [20]:
import numpy as np
from scipy.optimize._numdiff import approx_derivative

A = np.array([
    [10., 2., 1., 3., 0.],
    [2., 12., 2., 1., 4.],
    [1., 2., 15., 3., 2.],
    [3., 1., 3., 14., 5.],
    [0., 4., 2., 5., 13.]
])

b_vec = np.array([15., 20., 30., 25., 18.])

exact_solution = np.array([
    0.8204192602132074,
    1.0454174715604208,
    1.5387994562495528,
    1.0553909994991773,
    0.4202904772125635
])

exact_det = 209655.0

print_header("Result (norm)", "Linear System Method")

solvers = [
    ("Gauss", lambda: nm.linearsystem_solve(A, b_vec, method="gauss")),
    ("LU", lambda: nm.linearsystem_solve(A, b_vec, method="lu")),
    ("Cholesky", lambda: nm.linearsystem_solve(A, b_vec, method="cholesky")),
    ("QR", lambda: nm.linearsystem_solve(A, b_vec, method="QR")),
    ("NumPy", lambda: np.linalg.solve(A, b_vec)),
]

for name, solver in solvers:
    x, elapsed = timed(solver)
    error = np.linalg.norm(x - exact_solution)
    print_row(name, np.linalg.norm(x), error, elapsed)

print("-" * 75)

(L, U), elapsed = timed(nm.lu_decomposition, A)
error = np.linalg.norm(L @ U - A)
print_row("LU Decomposition", None, error, elapsed)

print("-" * 75)
print_header("Result (norm)", "Determinant Method")


det_methods = [
    ("Gauss", lambda: nm.determinant_calculate(A, method="gauss")),
    ("LU", lambda: nm.determinant_calculate(A, method="lu")),
    ("NumPy", lambda: np.linalg.det(A)),
]

for name, det_func in det_methods:
    det, elapsed = timed(det_func)
    error = abs(det - exact_det)
    print_row(name, det, error, elapsed)

print("-" * 75)


def F(v):
    x_, y_ = v
    return np.array([x_**2 + y_**2 - 4, x_ - y_])

point = np.array([1.0, 1.0])
h = 1e-6

exact_jacobian = np.array([
    [2.0, 2.0],
    [1.0, -1.0]
])

jacobian_methods = [
    ("Jacobian", lambda: nm.jacobian_calculate(F, point, h)),
    ("Jacobian (SciPy)", lambda: approx_derivative(F, point)),
]

for name, jac_func in jacobian_methods:
    J, elapsed = timed(jac_func)
    error = np.linalg.norm(J - exact_jacobian)
    print_row(name, None, error, elapsed)
    print(J)

Linear System Method            Result (norm)          Error      Time (ms)
---------------------------------------------------------------------------
Gauss                                2.329031       2.00e-16         0.1328
LU                                   2.329031       2.99e-16         0.0582
Cholesky                             2.329031       3.33e-16         0.1419
QR                                   2.329031       7.71e-16         0.1843
NumPy                                2.329031       3.72e-16         0.0387
---------------------------------------------------------------------------
LU Decomposition                            —       4.44e-16         0.0382
---------------------------------------------------------------------------
Determinant Method              Result (norm)          Error      Time (ms)
---------------------------------------------------------------------------
Gauss                           209655.000000       2.91e-11         0.0486
LU          